# Interactive 3-D Surface (implied volatility)

**Family:** Interactive, animated & big data &nbsp;|&nbsp; **Library:** plotly &nbsp;|&nbsp; **Level:** Advanced

> *Interactive, animated & big data:* What-if sliders, linked views, animation and million-point rendering. Use these when exploration or motion adds understanding that a static chart cannot.

## Design purpose

A rotatable 3-D surface of a measure over two continuous inputs, here implied volatility over strike and expiry, shown next to a 2-D heatmap of the same grid. The surface gives the overall shape (skew, smile, term structure) at a glance, and the heatmap is where values are actually read.

## When to use

- Show the shape of a function of two variables: volatility surfaces, yield/term surfaces, loss landscapes, response surfaces
- The reader can rotate and hover (interactive HTML, notebook)
- Shape matters more than exact values

## Avoid when

- Static images and print: a fixed 3-D view hides parts of the surface and distorts heights (use a heatmap or contour plot)
- Comparing precise values (perspective makes heights unreadable)
- Bars or pies in 3-D: never

## Alternatives

- Heatmap or contour plot (precise and static)
- Small multiples of 2-D slices (e.g. smiles by expiry)

## Design tips

- Always pair a 3-D view with a 2-D one that can be read precisely
- Use one sequential hue shared by both views and a single color bar
- Project contours onto the floor to connect the 3-D shape to 2-D positions
- Choose a default camera angle that shows the main feature (here, the downside skew)

## Example

Simulated equity-index implied volatility for strikes at 70–130% of spot and expiries of 1 week to 2 years: downside skew, a smile at short expiries, and an upward-sloping term structure.

### Setup

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
           "#e87ba4", "#008300", "#6250d6", "#e34948"]
pio.templates["vizlib"] = go.layout.Template(layout=dict(
    colorway=PALETTE, font=dict(family="Helvetica, Arial, sans-serif", size=13, color="#0b0b0b"),
    paper_bgcolor="#fcfcfb", plot_bgcolor="#fcfcfb", title=dict(x=0.02, xanchor="left"),
    xaxis=dict(gridcolor="#e6e5e1", linecolor="#c9c8c3", zeroline=False),
    yaxis=dict(gridcolor="#e6e5e1", linecolor="#c9c8c3", zeroline=False),
))
pio.templates.default = "plotly_white+vizlib"

from plotly.subplots import make_subplots
pio.renderers.default = "plotly_mimetype+notebook_connected"

### Data

In [2]:
moneyness = np.linspace(0.70, 1.30, 25)                     # strike / spot
expiry = np.array([1/52, 2/52, 1/12, 2/12, 3/12, 6/12, 9/12, 1.0, 1.5, 2.0])
K, T = np.meshgrid(moneyness, expiry)
x = np.log(K)
atm = 0.16 + 0.025 * np.log1p(4 * T)                          # term structure
skew = -0.32 / np.sqrt(T + 0.15) * x                          # downside skew, fading with expiry
smile = 0.9 / (T + 0.25) * x**2                                # curvature, strongest short-dated
iv = 100 * (atm + skew * 0.35 + smile * 0.35)
expiry_labels = ["1W", "2W", "1M", "2M", "3M", "6M", "9M", "1Y", "18M", "2Y"]
surface = pd.DataFrame(iv.round(1), index=expiry_labels, columns=(100 * moneyness).round().astype(int))
surface.iloc[:, ::4]

,70,80,90,100,110,120,130
1W,40.8,28.1,20.4,16.2,14.7,15.1,17.1
2W,39.5,27.6,20.3,16.4,14.9,15.3,17.1
1M,37.0,26.6,20.2,16.7,15.4,15.6,17.1
2M,34.0,25.5,20.2,17.3,16.1,16.2,17.3
3M,32.1,24.8,20.3,17.7,16.6,16.6,17.4
6M,29.0,23.9,20.7,18.7,17.8,17.6,18.0
9M,27.7,23.7,21.1,19.5,18.6,18.4,18.5
1Y,27.0,23.6,21.4,20.0,19.3,19.0,19.0
18M,26.3,23.7,22.0,20.9,20.2,19.9,19.8
2Y,26.0,23.9,22.5,21.5,20.9,20.6,20.5


### Plot

In [3]:
fig = make_subplots(rows=1, cols=2, column_widths=[0.58, 0.42], horizontal_spacing=0.06,
                    specs=[[{"type": "surface"}, {"type": "heatmap"}]],
                    subplot_titles=("Rotate me: shape", "Read me: values (%)"))
fig.add_trace(go.Surface(
    x=100 * moneyness, y=np.arange(len(expiry)), z=iv, coloraxis="coloraxis",
    contours=dict(z=dict(show=True, project_z=True, usecolormap=True, width=1)),
    hovertemplate="strike %{x:.0f}% · expiry %{customdata}<br>IV %{z:.1f}%<extra></extra>",
    customdata=np.tile(np.array(expiry_labels)[:, None], (1, len(moneyness)))), row=1, col=1)
fig.add_trace(go.Heatmap(
    x=100 * moneyness, y=expiry_labels, z=iv, coloraxis="coloraxis",
    hovertemplate="strike %{x:.0f}% · expiry %{y}<br>IV %{z:.1f}%<extra></extra>"), row=1, col=2)
fig.update_layout(
    title="Implied volatility surface (simulated equity index)", height=560,
    coloraxis=dict(colorscale="Blues", colorbar=dict(title="IV (%)", x=1.02)),
    scene=dict(xaxis_title="Strike (% of spot)", zaxis_title="IV (%)",
               yaxis=dict(title="Expiry", tickvals=list(range(len(expiry))), ticktext=expiry_labels),
               camera=dict(eye=dict(x=-1.6, y=-1.5, z=0.9))),
    margin=dict(l=10, r=10, t=80, b=40),
)
fig.update_xaxes(title_text="Strike (% of spot)", row=1, col=2)
fig.update_yaxes(title_text="Expiry", row=1, col=2)
fig.show()

## How to read it

Rotate the surface: volatility is highest for low strikes at short expiries (crash protection is expensive), and the skew flattens as expiry lengthens. Read exact values from the heatmap on the right, or hover on either view.

---
Standalone script: [`surface_3d_interactive.py`](surface_3d_interactive.py) &nbsp;·&nbsp; [Catalogue](../../README.md) &nbsp;·&nbsp; [Gallery](../../gallery/index.html)